# 06 - API submission

Building a digital twin is a means, not an end. Eventually you hand your scenario data to an external solver and get results back. The `backend.api_submission` package handles the *delivery*: it is a thin, UI-independent transport layer. A service is described by **where it listens** (an HTTP endpoint or a Redis stream) and **what shape it wants** (a requirements template); the transport just posts the payload there.

This notebook uses the bundled **`demo_energy_simulator`** - a small building-energy service that ships with the platform. It runs locally in Docker, uses synthetic weather, and needs no credentials or external accounts, so the whole flow works offline.

We will walk through:

1. Starting the demo service and checking it is reachable
2. The service's requirements template - what shape it wants
3. The payload Digicities builds from a scenario
4. Submitting it through the transport layer
5. Reading the results back

In [1]:
import os, sys, pathlib, json, urllib.request

REPO_ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

# The demo service is published on host port 5001 by docker-compose.
# Start it with:  docker compose up -d --build demo_energy_simulator
SERVICE_URL = os.environ.get(
    "DEMO_ENERGY_SIMULATOR_URL", "http://localhost:5001/api/energy_simulation"
)

def _demo_reachable(url):
    # The service exposes a public /api_docs endpoint; a quick GET tells us it is up.
    probe = url.rstrip("/") + "/api_docs"
    try:
        with urllib.request.urlopen(probe, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

DEMO_REACHABLE = _demo_reachable(SERVICE_URL)
if DEMO_REACHABLE:
    print(f"demo_energy_simulator is up at {SERVICE_URL} - live cells will run.")
else:
    print("demo_energy_simulator not reachable.")
    print("Start it with:  docker compose up -d --build demo_energy_simulator")
    print("The notebook still shows the template, payload, and transport API;")
    print("only the network calls are skipped.")

demo_energy_simulator is up at http://localhost:5001/api/energy_simulation - live cells will run.


## 6.1 The service template

A service tells Digicities what payload it wants through a **requirements template** - a YAML file that maps ontology attributes to payload fields. The demo workspace ships one at `demo_workspaces/energy-simulation/services/demo_energy_simulator.yaml`.

In [2]:
import yaml

template_path = REPO_ROOT / "demo_workspaces/energy-simulation/services/demo_energy_simulator.yaml"
template = yaml.safe_load(template_path.read_text(encoding="utf-8"))

print(f"service_name: {template['service_name']}")
print(f"description:  {template['description']}\n")
print("Payload template (payload field <- ontology term):")
print(yaml.safe_dump(template["scenario_data"], sort_keys=False))

service_name: demo_energy_simulator
description:  Bundled example. Estimates annual building energy demand (heating, hot water, electricity) per building.

Payload template (payload field <- ontology term):
uri: Scenario.URI
name: Scenario.label
location:
  link: CL.Scenario.Location
  template:
    uri: Location.URI
    weather_data: Location.WeatherEPW
    buildings:
      link: CL.Location.Building
      template:
        uri: Building.URI
        SIA2024BuildingType: Building.SIA2024BuildingType
        BuildingAge: Building.BuildingAge
        GroundFloorArea: Building.GroundFloorArea
        NumberOfFloors: Building.NumberOfFloors
        HeatingSupply: Building.HeatingSupply
        DHWSupply: Building.DHWSupply



## 6.2 The payload

Digicities builds the payload by walking your **scenario** with that template: it resolves each ontology term against the scenario's components and produces plain JSON. In the Streamlit UI this is the **Convert** tab (`ttl_convert`); it reads the scenario from the knowledge graph, applies the template, and validates the result before submission.

Below is the payload the converter produces for the demo workspace's baseline scenario. Note `location` is a **list** - a scenario can span several locations, and the demo service accepts one or many.

In [3]:
payload = {
    "service_name": "demo_energy_simulator",
    "description": template["description"],
    "scenario_data": {
        "uri": "https://digicities.info/proj/energy-simulation/EnergySimBaseline",
        "name": "Energy Sim - Baseline (MFH)",
        "location": [
            {
                "uri": "https://digicities.info/proj/energy-simulation/Location/TownCentre",
                "weather_data": "demo_weather.epw",
                "buildings": [
                    {
                        "uri": "https://digicities.info/proj/energy-simulation/Building/MFH_1",
                        "SIA2024BuildingType": "MFH",
                        "BuildingAge": "1985",
                        "GroundFloorArea": 284.0,
                        "NumberOfFloors": 4.0,
                        "HeatingSupply": "GasHeated",
                        "DHWSupply": "GasHeated",
                    },
                ],
            },
        ],
    },
}

n_buildings = sum(len(loc["buildings"]) for loc in payload["scenario_data"]["location"])
print(f"Payload for '{payload['scenario_data']['name']}': "
      f"{n_buildings} building(s) across "
      f"{len(payload['scenario_data']['location'])} location(s).")

Payload for 'Energy Sim - Baseline (MFH)': 1 building(s) across 1 location(s).


## 6.3 Submit through the transport layer

`backend.api_submission.transports.submit_http` POSTs the payload as JSON and treats a 2xx response as success. It is pure Python - no Streamlit, no solver-specific code. (For stream-based services there is `submit_redis`; the flexibility-optimizer use-case uses it.)

In [4]:
from backend.api_submission.transports import submit_http

if DEMO_REACHABLE:
    result = submit_http(payload, url=SERVICE_URL)
    print(f"success = {result.success}  (HTTP {result.status_code})")
    if result.success:
        summary = result.response_data.get("summary", {})
        print(f"scenario_id: {result.response_data.get('scenario_id')}")
        print(f"annual consumption: "
              f"{summary.get('total_annual_consumption_kWh'):,.0f} kWh "
              f"across {summary.get('number_of_buildings')} building(s)")
        print(f"dashboard: {result.response_data.get('result_url')}")
    else:
        print(f"error: {result.error_message}")
else:
    print("Skipping live submission - start the demo service (see cell 1) and re-run.")

success = True  (HTTP 200)
scenario_id: 75f9065e
annual consumption: 114,401 kWh across 1 building(s)
dashboard: http://localhost:5001/api/energy_simulation/view/75f9065e


## 6.4 Reading the results back

The response carries a `result_url` (an HTML dashboard you can open in a browser) and an `api_url` (the same results as JSON). The demo service persists results to a Docker volume, so the links keep working after submission. Let's pull the JSON.

In [5]:
if DEMO_REACHABLE and "result" in globals() and result.success:
    api_url = result.response_data.get("api_url")
    with urllib.request.urlopen(api_url, timeout=10) as r:
        full = json.load(r)
    agg = full.get("aggregate", {})
    print(f"weather source: {full.get('weather_source')} ({full.get('weather_location')})")
    print(f"total annual consumption: {agg.get('total_annual_consumption_kWh'):,.0f} kWh")
    print(f"peak demand: {agg.get('peak_demand_kW'):,.1f} kW\n")
    for b in full.get("buildings", []):
        meta = b["metadata"]
        print(f"  - {meta['building_type']} ({meta['total_area_m2']:.0f} m2): "
              f"{b['annual_consumption_kWh']:,.0f} kWh "
              f"({b['consumption_per_m2']:.0f} kWh/m2)")
else:
    print("No live result to fetch - run 6.3 against a running demo service first.")

weather source: file (Demo City)
total annual consumption: 114,401 kWh
peak demand: 23.6 kW

  - MFH (1136 m2): 114,401 kWh (101 kWh/m2)


## 6.5 Other services and transports

The transport layer is deliberately small, so wiring in another service is mostly configuration, not code:

- **HTTP service** - register its endpoint URL and a requirements template; `submit_http` delivers the payload. That is exactly what we did here.
- **Stream service** - for services that consume a Redis request stream (and optionally publish results back), use `submit_redis(payload, request_stream=..., result_stream=...)`. The flexibility-optimizer use-case works this way.

In the Streamlit UI you register a service on the **API Configuration** tab (or, for the demo, just click **Connect**), then **Convert** a scenario and **Submit**. Everything here is what the UI does under the hood, so batch runs, experiments, or your own solver integration can call `backend.api_submission` directly.

## That's the tutorial

You have now seen the full shape of the platform from the Python side:

1. **Ontology** - the vocabulary everything is built on
2. **Replica builder** - constructing digital twins
3. **Scenario builder** - pulling a baseline back out
4. **Assumptions** - deriving modified scenarios
5. **Data products** - parsing shipped TTL
6. **API submission** - handing a scenario to an external solver and reading results back

The Streamlit UI automates all of this, but for anything the UI does not do - batch runs, experiments, custom solvers, integration with your own models - go straight for the `backend/` package. Every function you have seen in the notebooks is a supported public API.